In [2]:
# =============================================================================
# INDIVIDUAL METHOD RUNNER
# Quick testing of a single method on a single dataset
# =============================================================================

# -----------------------------------------------------------------------------
# CONFIGURATION - CHANGE THESE VALUES
# -----------------------------------------------------------------------------

METHOD = "modernNCA"           # Method to run (e.g., 'xgboost', 'catboost', 'tabpfn', 'mlp')
DATASET = "0006.lgd_freddie"        # Dataset name (e.g., '0014.hmeq', '0001.gmsc')
TASK = "lgd"                  # Task type: 'pd' (classification) or 'lgd' (regression)

# -----------------------------------------------------------------------------
# FIXED SETTINGS (for quick testing)
# -----------------------------------------------------------------------------

ROW_LIMIT = 10000             # Limit rows for fast execution
MAX_EPOCHS = 15              # Max epochs for deep learning methods
CV_SPLITS = 1                # Single fold
TUNE = False                 # No HPO
SEED = 42                    # Random seed
TEST_SIZE = 0.2              # Test set fraction
VAL_SIZE = 0.2               # Validation set fraction

# -----------------------------------------------------------------------------
# SETUP
# -----------------------------------------------------------------------------

import sys
from pathlib import Path
import pickle
import json
from datetime import datetime

# Add project root to path (notebook is in notebooks/ folder)
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")
print(f"\n{'='*60}")
print(f" Running: {METHOD} on {DATASET} ({TASK.upper()})")
print(f"{'='*60}")
print(f"  Row limit:  {ROW_LIMIT}")
print(f"  Max epochs: {MAX_EPOCHS}")
print(f"  CV splits:  {CV_SPLITS}")
print(f"  HPO:        {TUNE}")
print(f"{'='*60}\n")

# -----------------------------------------------------------------------------
# RUN METHOD
# -----------------------------------------------------------------------------

from src.methods.method_runner import run_talent_method, get_available_methods

# Show available methods
available = get_available_methods()
print(f"Available classical methods: {available['classical']}")
print(f"Available deep methods: {available['deep'][:10]}... ({len(available['deep'])} total)")
print()

# Run the method
results = run_talent_method(
    task=TASK,
    dataset=DATASET,
    test_size=TEST_SIZE,
    val_size=VAL_SIZE,
    cv_splits=CV_SPLITS,
    seed=SEED,
    row_limit=ROW_LIMIT,
    method=METHOD,
    max_epoch=MAX_EPOCHS,
    tune=TUNE,
    verbose=True,
)

# -----------------------------------------------------------------------------
# DISPLAY RESULTS
# -----------------------------------------------------------------------------

print(f"\n{'='*60}")
print(f" RESULTS")
print(f"{'='*60}")

for fold_id, fold_results in results.items():
    print(f"\nFold {fold_id}:")
    print(f"  Train time: {fold_results['train_time']:.2f}s")
    print(f"  Samples:    {len(fold_results['y_true'])}")
    
    if TASK == 'lgd':
        print(f"  Clipped:    {fold_results['n_clipped_below']} below, {fold_results['n_clipped_above']} above")
    
    print(f"\n  Metrics:")
    for metric_name, metric_value in fold_results['metrics'].items():
        if not (isinstance(metric_value, float) and metric_value != metric_value):  # Skip NaN
            print(f"    {metric_name:20s}: {metric_value:.4f}")

# -----------------------------------------------------------------------------
# SAVE RESULTS
# -----------------------------------------------------------------------------

# Create output directory
output_dir = PROJECT_ROOT / 'results' / 'individual_method_runner'
output_dir.mkdir(parents=True, exist_ok=True)

# Generate filename with timestamp
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
filename = f"{METHOD}_{DATASET}_{TASK}_{timestamp}"

# Save as pickle (full results)
pickle_path = output_dir / f"{filename}.pkl"
with open(pickle_path, 'wb') as f:
    pickle.dump(results, f)
print(f"\nResults saved to: {pickle_path}")

# Save summary as JSON (metrics only, for easy viewing)
summary = {
    'method': METHOD,
    'dataset': DATASET,
    'task': TASK,
    'timestamp': timestamp,
    'config': {
        'row_limit': ROW_LIMIT,
        'max_epochs': MAX_EPOCHS,
        'cv_splits': CV_SPLITS,
        'tune': TUNE,
        'seed': SEED,
    },
    'folds': {}
}

for fold_id, fold_results in results.items():
    summary['folds'][fold_id] = {
        'train_time': fold_results['train_time'],
        'n_samples': len(fold_results['y_true']),
        'metrics': {k: v for k, v in fold_results['metrics'].items() if not (isinstance(v, float) and v != v)},
    }
    if TASK == 'lgd':
        summary['folds'][fold_id]['n_clipped_below'] = fold_results['n_clipped_below']
        summary['folds'][fold_id]['n_clipped_above'] = fold_results['n_clipped_above']

json_path = output_dir / f"{filename}.json"
with open(json_path, 'w') as f:
    json.dump(summary, f, indent=2)
print(f"Summary saved to: {json_path}")

print(f"\n{'='*60}")
print(f" DONE")
print(f"{'='*60}")

Project root: c:\Users\U0152019\OneDrive - KU Leuven\PhD Documents\Projects\1. TabPFN\TabPFNCredit

 Running: modernNCA on 0006.lgd_freddie (LGD)
  Row limit:  10000
  Max epochs: 15
  CV splits:  1
  HPO:        False

Available classical methods: ['LinearRegression', 'LogReg', 'NCM', 'NaiveBayes', 'RandomForest', 'catboost', 'dummy', 'knn', 'lightgbm', 'svm', 'xgboost']
Available deep methods: ['amformer', 'autoint', 'bishop', 'danets', 'dcn2', 'dnnr', 'excelformer', 'ftt', 'grande', 'grownet']... (38 total)


Running modernNCA (deep) on 0006.lgd_freddie (LGD)

Preparing data with 1 CV splits...
Fold IDs: [1]
First fold ID: 1

Directory setup:
  Config directory (persistent): C:\Users\U0152019\OneDrive - KU Leuven\PhD Documents\Projects\1. TabPFN\TabPFNCredit\config_hpo\lgd\0006.lgd_freddie\modernNCA\NO_HPO
  Checkpoint directory (temp):   C:\Users\U0152019\AppData\Local\Temp\talent_ckpt_0006.lgd_freddie_modernNCA_i_cmi6ny

[HPO] Mode: DISABLED
[HPO] All folds: Will use TALENT's defa

2it [00:00, 14.04it/s]

epoch 0, val, loss=0.6945 regression result=0.2173
Epoch: 0, Time cost: 0.9222095012664795


epoch 1, train 1/7, loss=0.6722 lr=0.01
best epoch 0, best val res=0.2173


2it [00:00, 14.01it/s]

...

epoch 14, train 1/7, loss=0.5139 lr=0.01
best epoch 9, best val res=0.2011


2it [00:00, 12.16it/s]

epoch 14, val, loss=0.6507 regression result=0.2032
Epoch: 14, Time cost: 1.025465726852417
best epoch 9, best val res=0.2011



2it [00:00, 10.59it/s]


Test: loss=0.6578
[MAE]=0.2043
[R2]=0.3591
[RMSE]=0.2560

Fold 1 metrics:
  R2: 0.3591
  MSE: 0.0655
  RMSE: 0.2560
  MAE: 0.2043
  MedAE: 0.1743
  MaxError: 0.9635
  Explained_Variance: 0.3591
  MAPE: 191.3494
  Pearson_Corr: 0.5994
  Spearman_Corr: 0.6043

Completed 1 folds for modernNCA


 RESULTS

Fold 1:
  Train time: 14.86s
  Samples:    2000
  Clipped:    0 below, 0 above

  Metrics:
    R2                  : 0.3591
    MSE                 : 0.0655
    RMSE                : 0.2560
    MAE                 : 0.2043
    MedAE               : 0.1743
    MaxError            : 0.9635
    Explained_Variance  : 0.3591
    MAPE                : 191.3494
    Pearson_Corr        : 0.5994
    Spearman_Corr       : 0.6043

Results saved to: c:\Users\U0152019\OneDrive - KU Leuven\PhD Documents\Projects\1. TabPFN\TabPFNCredit\results\individual_method_runner\modernNCA_0006.lgd_freddie_lgd_20251223_133114.pkl
Summary saved to: c:\Users\U0152019\OneDrive - KU Leuven\PhD Documents\Projects\1. Tab